# 演習② DPO（穴埋め形式）

`# TODO:` のコメント箇所を自分で実装してください。

**目標**:
- 自分で preference データを 3 件以上作成する
- DPOConfig の beta パラメータを変えて出力の変化を観察する

In [ ]:
import os
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model
from trl import DPOTrainer, DPOConfig

os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# TODO: 自分で preference データを 3 件以上作成してください
# 条件:
#   - 各エントリに 'prompt', 'chosen', 'rejected' キーを含める
#   - chosen と rejected の差が明確なものにする
#   - テーマは自由（技術的な質問、文章スタイル、説明の丁寧さ など）

my_preference_data = [
    # TODO: ここに 3 件以上追加する
    # 例:
    # {
    #     'prompt': '...',
    #     'chosen': '...',  # 良い回答
    #     'rejected': '...',  # 悪い回答
    # },
]

assert len(my_preference_data) >= 3, 'データは 3 件以上作成してください'
dataset = Dataset.from_list(my_preference_data)
print(f'作成したデータ: {len(dataset)} 件')

In [ ]:
# TODO: DPOConfig を設定してください
# 条件:
#   - output_dir='./outputs/ex02_dpo'
#   - max_steps=20
#   - beta=0.1（後で 0.05 と 0.3 でも試す）
#   - max_length=512, max_prompt_length=256

BETA = 0.1  # ここを変えて実験しよう

BASE_MODEL = 'meta-llama/Meta-Llama-3-8B'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
) if device == 'cuda' else None

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto' if device == 'cuda' else None,
    torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)

dpo_args = None  # TODO: DPOConfig(...) で置き換える

trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
trainer.train()
print('DPO 学習完了！')

## 実験課題

1. beta=0.05, 0.1, 0.3 の 3 パターンで DPO を実行し、同じプロンプトへの出力を比較してください
2. 自分が作成した chosen と rejected のペアで、モデルの出力がどちらに近づいたか確認してください

解答は `solutions/sol_02_dpo.ipynb` を参照してください。